###### ============================================================
##### HOTEL BOOKING DATA QUALITY & VALIDATION PIPELINE
###### ============================================================
######
##### Purpose:
##### This notebook performs data profiling, cleaning, validation, and quality assessment on hotel booking datasets.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
NOTEBOOK_DIR = Path.cwd() # Get the folder containing this notebook 

PROJECT_DIR = NOTEBOOK_DIR.parent

# Define important directories

RAW_DATA_DIR = PROJECT_DIR/"data"/"raw"
PROCESSED_DATA_DIR = PROJECT_DIR/"data"/"processed"
QUARANTINE_DIR = PROJECT_DIR/"data"/"quarantine"
REPORTS_DIR = PROJECT_DIR/"reports"
PARQUET_DATA_DIR = PROJECT_DIR/"data"/"parquet"

print(RAW_DATA_DIR)
print(PROCESSED_DATA_DIR)
print(QUARANTINE_DIR)
print(REPORTS_DIR)

d:\datacleaning_pipeline\data_cleaning\data\raw
d:\datacleaning_pipeline\data_cleaning\data\processed
d:\datacleaning_pipeline\data_cleaning\data\quarantine
d:\datacleaning_pipeline\data_cleaning\reports


In [3]:
# List all CSV files available in the raw data directory.

raw_files = list(RAW_DATA_DIR.glob("*.csv"))

print(f"Number of csv files: {len(raw_files)}")

print("\nFiles found:")
for file in raw_files:
    print(f"- {file.name}")


Number of csv files: 5

Files found:
- dim_date.csv
- dim_hotels.csv
- dim_rooms.csv
- fact_aggregated_bookings.csv
- fact_bookings.csv


In [4]:
# ============================================================
# LOAD RAW DATASETS
# ============================================================

# Define the raw CSV files we expect in the project.
# Dictionary key   → table name we will use in Python
# Dictionary value → corresponding CSV filename

raw_files = {
    "fact_bookings": "fact_bookings.csv",
    "fact_aggregated_bookings": "fact_aggregated_bookings.csv",
    "dim_hotels": "dim_hotels.csv",
    "dim_rooms": "dim_rooms.csv",
    "dim_date": "dim_date.csv"
}

datasets = {}

for table_name,filename in raw_files.items():
    file_path = RAW_DATA_DIR/filename
    datasets[table_name] = pd.read_csv(file_path)

print(f"Successfully loaded {len(datasets)} tables")

print("\nLoaded datasets:")

for table_name in datasets:
    print(f"- {table_name}")



Successfully loaded 5 tables

Loaded datasets:
- fact_bookings
- fact_aggregated_bookings
- dim_hotels
- dim_rooms
- dim_date


In [5]:
# ============================================================
# DATASET OVERVIEW
# ============================================================

# Create a summary of all loaded datasets.

dataset_overview = []

for table_name, df in datasets.items():
    dataset_overview.append({
        "table_name":table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "total_missing_values": df.isna().sum().sum()

    })


dataset_overview = pd.DataFrame(dataset_overview)

dataset_overview

,table_name,rows,columns,duplicate_rows,total_missing_values
0,fact_bookings,134590,12,0,77907
1,fact_aggregated_bookings,9200,5,0,0
2,dim_hotels,25,4,0,0
3,dim_rooms,4,2,0,0
4,dim_date,92,4,0,0


In [6]:
# ============================================================
# COLUMN-LEVEL STRUCTURE
# ============================================================

for table_name, df in datasets.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")

    print("\nColumn names and data types:")

    display(
        pd.DataFrame({
            "column_name":df.columns,
            "data_type":df.dtypes.astype(str).values
        })
    )
    


TABLE: fact_bookings
Rows: 134,590
Columns: 12

Column names and data types:


,column_name,data_type
0,booking_id,object
1,property_id,int64
2,booking_date,object
3,check_in_date,object
4,checkout_date,object
5,no_guests,int64
6,room_category,object
7,booking_platform,object
8,ratings_given,float64
9,booking_status,object



TABLE: fact_aggregated_bookings
Rows: 9,200
Columns: 5

Column names and data types:


,column_name,data_type
0,property_id,int64
1,check_in_date,object
2,room_category,object
3,successful_bookings,int64
4,capacity,int64



TABLE: dim_hotels
Rows: 25
Columns: 4

Column names and data types:


,column_name,data_type
0,property_id,int64
1,property_name,object
2,category,object
3,city,object



TABLE: dim_rooms
Rows: 4
Columns: 2

Column names and data types:


,column_name,data_type
0,room_id,object
1,room_class,object



TABLE: dim_date
Rows: 92
Columns: 4

Column names and data types:


,column_name,data_type
0,date,object
1,mmm yy,object
2,week no,object
3,day_type,object


In [ ]:
# ============================================================
# SOURCE ROW LINEAGE
# ============================================================

# Assign every source record a permanent identifier.
#
# This identifier allows us to trace a quarantined record
# back to its original position in the raw source file.
#
# The identifier is created before any cleaning or filtering.

for table_name,df in datasets.items():
    df = df.copy()

    # Only create the column if it does not already exist.
    if '_source_row_id' not in df.columns:
        df.insert(0,"_source_row_id",np.arange(1,len(df) + 1))

    datasets[table_name] = df

print("Source row lineage added successfully.")



Source row lineage added successfully.


In [8]:
# ============================================================
# MISSING VALUE PROFILING
# ============================================================

missing_profiles = []

for table_name,df in datasets.items():

    
    missing_count = df.isna().sum() # Calculate missing values for every column

    missing_percentage = df.isna().mean()*100 # Calculate missing percentage for every column
    

    #Build a record for every table

    for column in df.columns:
        missing_profiles.append({
            "table_name": table_name,
            "column_name": column,
            "missing_count": missing_count[column],
            "missing_percentage": round(missing_percentage[column],2)
        })


missing_profile = pd.DataFrame(missing_profiles) #Convert results into a dataframe

#Display columns having missing values
missing_profile[missing_profile["missing_count"]>0].sort_values(by="missing_percentage",ascending = False)

,table_name,column_name,missing_count,missing_percentage
9,fact_bookings,ratings_given,77907,57.88


In [9]:
# ============================================================
# BLANK / WHITESPACE VALUE PROFILING
# ============================================================

blank_profiles = []

for table_name,df in datasets.items():

    for column in df.select_dtypes(include = 'object').columns:

        blank_count = df[column].fillna("").astype(str).str.strip().eq("").sum()

        blank_profiles.append({
            "table_name":table_name,
            "column_name":column,
            "blank_or_whitespace_count":blank_count
        })

blank_profile = pd.DataFrame(blank_profiles)

#Display columns containing only blank values

blank_profile[blank_profile["blank_or_whitespace_count"]>0].sort_values(by="blank_or_whitespace_count",ascending = False)
    

,table_name,column_name,blank_or_whitespace_count


In [10]:
# ============================================================
# DUPLICATE ROW PROFILING
# ============================================================

duplicate_profiles = []

for table_name,df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_profiles.append({
        "table_name":table_name,
        "total_rows":len(df),
        "duplicate_rows":duplicate_count,
        "duplicate_percentage":round((duplicate_count/len(df))*100,2)
    })

duplicate_profile = pd.DataFrame(duplicate_profiles)

duplicate_profile


,table_name,total_rows,duplicate_rows,duplicate_percentage
0,fact_bookings,134590,0,0.00
1,fact_aggregated_bookings,9200,0,0.00
2,dim_hotels,25,0,0.00
3,dim_rooms,4,0,0.00
4,dim_date,92,0,0.00


In [11]:
# ============================================================
# BUSINESS KEY DUPLICATE CHECK (Since there are no completely identical rows)
# ============================================================

# ------------------------------------------------------------
# 1. FACT_BOOKINGS
# Business key: booking_id
# ------------------------------------------------------------

booking_df = datasets['fact_bookings']

duplicate_bookings = (booking_df[booking_df['booking_id'].duplicated(keep=False)].sort_values('booking_id'))

print("=" * 70)
print("DUPLICATE BOOKING IDs - FACT_BOOKINGS")
print("=" * 70)

print(f"Rows involved in duplicated booking IDs: {len(duplicate_bookings):,}")

display(duplicate_bookings.head(20))

# ------------------------------------------------------------
# 2. FACT_AGGREGATED_BOOKINGS
# Business key:
# property_id + check_in_date + room_category
# ------------------------------------------------------------

aggregated_df = datasets["fact_aggregated_bookings"]

aggregated_key = ["property_id","check_in_date","room_category"]

duplicate_aggregated = aggregated_df[aggregated_df.duplicated(subset = aggregated_key,keep = False)].sort_values(aggregated_key)

print("\n" + "=" * 70)
print("DUPLICATE BUSINESS KEYS - FACT_AGGREGATED_BOOKINGS")
print("=" * 70)

# print(f"Rows involved in duplicated keys: "
#       f"{len(duplicate_aggregated):,}")

print(f"Rows involved in duplicated keys: {len(duplicate_aggregated):,}")

display(duplicate_aggregated.head(20))


DUPLICATE BOOKING IDs - FACT_BOOKINGS
Rows involved in duplicated booking IDs: 0


,_source_row_id,booking_id,property_id,booking_date,check_in_date,checkout_date,no_guests,room_category,booking_platform,ratings_given,booking_status,revenue_generated,revenue_realized



DUPLICATE BUSINESS KEYS - FACT_AGGREGATED_BOOKINGS
Rows involved in duplicated keys: 0


,_source_row_id,property_id,check_in_date,room_category,successful_bookings,capacity


In [ ]:
# ============================================================
# REUSABLE DUPLICATE VALIDATION FUNCTION
# ============================================================

def profile_duplicates(df,table_name,key_columns):
    """
    Identify duplicate records based on the specified business key.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset being validated.

    table_name : str
        Name of the dataset.

    key_columns : list
        Columns that define the expected unique business key.

    Returns
    -------
    dict
        Summary of duplicate quality checks.
    """

    # Check whether the specified key columns exist.
    missing_key_columns = []
    for column in key_columns:
        if column not in df.columns:
            missing_key_columns.append(column)

    if missing_key_columns:  #Checks whether list is empty or not
        raise ValueError(
            f"{table_name}: Missing key columns: {missing_key_columns}"  #Stops the program and raises error
        )

    #Identify every row involved in a duplicate key
    duplicate_mask = df.duplicated(subset = key_columns,keep = False)  

    duplicate_rows = df.loc[duplicate_mask].copy()  #Selects duplicated rows using boolean conditions. 
                                                   #Creates a separate dataframe
                                                   #Returns rows where duplicate mask is true
    
    #Count how many distinct keys are duplicated
    duplicated_key_count = df.loc[duplicate_mask,key_columns].drop_duplicates().shape[0]

    # duplicate_percentage = (duplicate_mask.sum()/len(df))*100
    result = {
        "table_name": table_name,
        "key_columns": ', '.join(key_columns),
        "total_rows": len(df),
        "duplicate_rows": duplicate_mask.sum(),
        "duplicate_keys": duplicated_key_count,
        "duplicate_percentage": round(duplicate_mask.mean()*100,2)

        }

    return result,duplicate_rows

In [13]:
# ============================================================
# APPLY DUPLICATE VALIDATION
# ============================================================

duplicate_results = []

# ------------------------------------------------------------
# FACT_BOOKINGS
# Unique key: booking_id
# ------------------------------------------------------------

booking_result,duplicate_bookings = profile_duplicates(datasets["fact_bookings"],"fact_bookings",["booking_id"])

duplicate_results.append(booking_result)

# ------------------------------------------------------------
# FACT_AGGREGATED_BOOKINGS
# Unique key:
# property_id + check_in_date + room_category
# ------------------------------------------------------------

aggregated_result,duplicate_aggregated = profile_duplicates(
    datasets['fact_aggregated_bookings'],'fact_aggregated_bookings',
    ['property_id','check_in_date','room_category']
)

duplicate_results.append(aggregated_result)

duplicate_quality_report = pd.DataFrame(
    duplicate_results
)

duplicate_quality_report

,table_name,key_columns,total_rows,duplicate_rows,duplicate_keys,duplicate_percentage
0,fact_bookings,booking_id,134590,0,0,0.00
1,fact_aggregated_bookings,"property_id, check_in_date, room_category",9200,0,0,0.00


In [14]:
# ============================================================
# QUARANTINE DUPLICATE RECORDS
# ============================================================

def quarantine_duplicates(df,table_name,key_columns,quarantine_dir):
    """
    Identify duplicate business-key records and save them
    separately for investigation.

    The original DataFrame is not modified.

    Returns
    -------
    cleaned_df : pandas.DataFrame
        DataFrame with duplicate-key records removed.

    duplicate_df : pandas.DataFrame
        Records quarantined for investigation.
    """

    duplicate_mask = df.duplicated(subset = key_columns,keep = False)

    duplicate_df = df.loc[duplicate_mask].copy()

    # If there are no duplicates, return the original
    # DataFrame unchanged.

    if duplicate_df.empty:
        print(f'{table_name}: No duplicate business keys found')

        return df.copy(), duplicate_df
    
    quarantine_path = (QUARANTINE_DIR/f'{table_name}_duplicate_records.csv')

    duplicate_df.to_csv(quarantine_path,index = False)

    # Remove all records belonging to duplicated keys
    # from the working dataset.
    #
    # We do NOT arbitrarily keep the first/last record,
    # because we don't yet know which record is correct.

    cleaned_df = df.loc[~duplicate_mask].copy()

    print(f'{table_name}: {len(duplicate_df):,} records quarantined')

    print(f'Quarantine file: {quarantine_path}')

    return cleaned_df,duplicate_df

In [15]:
# # ============================================================
# # DUPLICATE QUARANTINE PROCESS
# # ============================================================

# # ------------------------------------------------------------
# # FACT_BOOKINGS
# # ------------------------------------------------------------

fact_bookings_clean,fact_bookings_duplicates = (
    quarantine_duplicates(datasets['fact_bookings'],'fact_bookings',['booking_id'],QUARANTINE_DIR)
)

# # ------------------------------------------------------------
# # FACT_AGGREGATED_BOOKINGS
# # ------------------------------------------------------------

fact_aggregated_clean, fact_aggregated_duplicates = (
    quarantine_duplicates(datasets['fact_aggregated_bookings'],'fact_aggregated_bookings',['property_id','check_in_date','room_category'],
    QUARANTINE_DIR)
)



fact_bookings: No duplicate business keys found
fact_aggregated_bookings: No duplicate business keys found


In [16]:
# ============================================================
# DATE COLUMN VALIDATION
# ============================================================

date_columns = {
    'fact_bookings':[
        'booking_date',
        'check_in_date',
        'checkout_date'
    ],

    'fact_aggregated_bookings':[
        'check_in_date'
    ],

    'dim_date':[
        'date'
    ]
}

date_columns

{'fact_bookings': ['booking_date', 'check_in_date', 'checkout_date'],
 'fact_aggregated_bookings': ['check_in_date'],
 'dim_date': ['date']}

In [17]:
# ============================================================
# REUSABLE DATE VALIDATION FUNCTION
# ============================================================

def validate_date_column(df,table_name,column_name):
    """
    Validate one expected date column.

    The function does not modify the original DataFrame.

    It distinguishes between:

    1. Legitimate missing values
    2. Invalid non-null date values
    3. Successfully parsed dates

    Returns
    -------
    result : dict
        Validation summary for the column.

    invalid_mask : pandas.Series
        Boolean mask identifying non-null values that
        could not be interpreted as dates.
    """

    # --------------------------------------------------------
    # Check whether the expected column exists
    # --------------------------------------------------------

    if column_name not in df.columns:
        result = {
            'table_name': table_name,
            'column_name': column_name,
            'status': 'FAIL',
            'issue_type': 'missing_expected_column',
            'total_rows': len(df),
            'missing_values': None,
            'invalid_values': None,
            'valid_values': None
        }

        # No column exists, so there are no rows to return.
        invalid_mask = pd.Series(False,index = df.index)

        return result,invalid_mask

    converted_dates = pd.to_datetime(df[column_name],errors = 'coerce')
    
    # --------------------------------------------------------
    # Identify original missing values
    # --------------------------------------------------------

    missing_mask = df[column_name].isna()

    missing_count = int(missing_mask.sum())


    # --------------------------------------------------------
    # Identify invalid values
    #
    # A value is invalid when:
    #   - original value is NOT missing
    #   - conversion resulted in NaT
    # --------------------------------------------------------

    invalid_mask = (converted_dates.isna() & ~missing_mask)

    invalid_count = int(invalid_mask.sum())

    # --------------------------------------------------------
    # Valid values
    # --------------------------------------------------------

    valid_count = len(df) - missing_count - invalid_count

    # --------------------------------------------------------
    # Validation status
    # --------------------------------------------------------

    status = (
        'FAIL' 
        if invalid_count>0
        else 'PASS'
    )

    result = {
        'table_name': table_name,
        'column_name': column_name,
        'status': status,
        'issue_type': 
                'inavlid_date_values'
                if invalid_count>0
                else 'none',
        'total_rows': len(df),
        'missing_values': missing_count,
        'invalid_values': invalid_count,
        'valid_values': valid_count

    }
    return result,invalid_mask
         


In [18]:
# ============================================================
# RUN DATE VALIDATION ACROSS ALL TABLES
# ============================================================

date_validation_results = []
invalid_date_records = [] #list of dataframes containing invalid date records

for table_name,columns in date_columns.items():
    df = datasets[table_name]
    for column_name in columns:
        result,invalid_mask = validate_date_column(df,table_name,column_name)

        date_validation_results.append(result)

        # ----------------------------------------------------
        # If invalid values exist, capture them for
        # investigation/quarantine.
        # ----------------------------------------------------

        if invalid_mask.any():
            invalid_rows = df.loc[invalid_mask].copy() 
            invalid_rows['validation_table'] = table_name
            invalid_rows['validation_column'] = column_name
            invalid_rows['validation_rule'] = 'invalid_date_value'
            invalid_date_records.append(invalid_rows)


# Create the date validation report.
date_validation_report = pd.DataFrame(date_validation_results)
date_validation_report

C:\Users\Home\AppData\Local\Temp\ipykernel_12260\2780878292.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted_dates = pd.to_datetime(df[column_name],errors = 'coerce')
C:\Users\Home\AppData\Local\Temp\ipykernel_12260\2780878292.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted_dates = pd.to_datetime(df[column_name],errors = 'coerce')


,table_name,column_name,status,issue_type,total_rows,missing_values,invalid_values,valid_values
0,fact_bookings,booking_date,PASS,none,134590,0,0,134590
1,fact_bookings,check_in_date,PASS,none,134590,0,0,134590
2,fact_bookings,checkout_date,PASS,none,134590,0,0,134590
3,fact_aggregated_bookings,check_in_date,PASS,none,9200,0,0,9200
4,dim_date,date,PASS,none,92,0,0,92


In [19]:
# ============================================================
# INSPECT INVALID DATE RECORDS
# ============================================================

if invalid_date_records:
    invalid_date_df = pd.concat(invalid_date_records,ignore_index = True)

    print(f'Invalid date records found: {len(invalid_date_df):,}')
    display(invalid_date_df.head(20))

else:
    invalid_date_df = pd.DataFrame()

    print('No invalid date values were found')


No invalid date values were found


In [20]:
# ============================================================
# QUARANTINE INVALID DATE RECORDS
# ============================================================

def quarantine_invalid_date_records(invalid_date_records,quarantine_dir):
    """
    Quarantine records containing invalid date values.

    Only genuinely invalid, non-null date values should reach
    this function.

    Original source DataFrames are never modified.

    Parameters
    ----------
    invalid_date_records : list
        List of DataFrames containing invalid date records.

    quarantine_dir : pathlib.Path  
        Directory where quarantine files will be saved.

    Returns
    -------
    quarantined_records : pandas.DataFrame
        Combined DataFrame containing all quarantined records.
    """
    # --------------------------------------------------------
    # No invalid records
    # --------------------------------------------------------

    if not invalid_date_records:
        print('No quarantine required')
        return

    quarantined_records = []

    # --------------------------------------------------------
    # Process each table/date-column combination
    # --------------------------------------------------------

    for invalid_df in invalid_date_records:
        # Make a copy so that we never modify the original
        # DataFrame stored in memory.
        quarantine_df = invalid_df.copy()

        # ----------------------------------------------------
        # Add audit information
        # ----------------------------------------------------

        quarantine_df['quarantine_reason'] = 'invalid_date_value'
        quarantine_df['quarantine_timestamp'] = pd.Timestamp.now()

        # ----------------------------------------------------
        # Create a separate quarantine file for each
        # table + date column
        # ----------------------------------------------------

        output_file = quarantine_dir/f'{table_name}_{column_name}_invalid_dates.csv'

        quarantine_df.to_csv(output_file,index = False)

        print(f'{table_name}.{column_name}: {len(quarantine_df):,}')

        print('Saved to {output_file}')

        # Store for final combined report
        quarantined_records.append(quarantine_df)

        # --------------------------------------------------------
        # Combine all quarantined date records into one DataFrame
        # --------------------------------------------------------

        quarantined_records = pd.concat(quarantined_records)

        print("\n" + "=" * 70)
        print("INVALID DATE QUARANTINE COMPLETE")
        print("=" * 70)

        print(f'Total quarantined records: {len(quarantined_records):,}')

        return quarantined_records
        

In [21]:
# ============================================================
# RUN INVALID DATE QUARANTINE
# ============================================================

quarantined_invalid_dates = quarantine_invalid_date_records(invalid_date_records,QUARANTINE_DIR)

No quarantine required


In [22]:
# ============================================================
# NUMERIC COLUMN CONFIGURATION
# ============================================================

# Each table contains the numeric columns we expect.
#
# For each column we define:
#
#   minimum
#       Minimum acceptable business value.
#
#   maximum
#       Maximum acceptable business value.
#
#   allow_negative
#       Whether negative values are acceptable.

numeric_rules = {
     "fact_bookings": {

        "no_guests": {
            "minimum": 1,
            "maximum": 20
        },

        "ratings_given": {
            "minimum": 1,
            "maximum": 5
        },

        "revenue_generated": {
            "minimum": 0,
            "maximum": None
        },

        "revenue_realized": {
            "minimum": 0,
            "maximum": None
        }
    },

     "fact_aggregated_bookings": {

        "capacity": {
            "minimum": 0,
            "maximum": None
        },

        "successful_bookings": {
            "minimum": 0,
            "maximum": None
        }
    }
}

numeric_rules

{'fact_bookings': {'no_guests': {'minimum': 1, 'maximum': 20},
  'ratings_given': {'minimum': 1, 'maximum': 5},
  'revenue_generated': {'minimum': 0, 'maximum': None},
  'revenue_realized': {'minimum': 0, 'maximum': None}},
 'fact_aggregated_bookings': {'capacity': {'minimum': 0, 'maximum': None},
  'successful_bookings': {'minimum': 0, 'maximum': None}}}

In [23]:
# ============================================================
# REUSABLE NUMERIC VALIDATION FUNCTION
# ============================================================

def validate_numeric_column(df,table_name,column_name,minimum = None,maximum = None):
    """
    Validate a single numeric column.

    The original DataFrame is never modified.

    Validation distinguishes between:

    1. Missing values
    2. Non-numeric values
    3. Numeric values outside the allowed business range
    4. Valid numeric values

    Returns
    -------
    result : dict
        Column-level validation summary.

    invalid_mask : pandas.Series
        Rows containing invalid numeric values.
    """


    # --------------------------------------------------------
    # Check that the expected column exists.
    # --------------------------------------------------------

    if column_name not in df.columns:
        result = {
            "table_name": table_name,
            "column_name": column_name,
            "status": "FAIL",
            "issue_type": "missing_expected_column",
            "total_rows": len(df),
            "missing_values": None,
            "non_numeric_values": None,
            "out_of_range_values": None,
            "valid_values": None
        }

        invalid_mask = pd.Series(False,index = df.index) #every row is considered invalid as expected column is absent

        return result,invalid_mask

    # --------------------------------------------------------
    # Identify missing values.
    # We treat NaN, empty strings and whitespace as missing.
    # --------------------------------------------------------

    missing_mask = df[column_name].isna() | df[column_name].astype('string').str.strip().eq('')
    missing_count = missing_mask.sum()

    # --------------------------------------------------------
    # Temporarily convert values to numeric.
    # Invalid strings become NaN.
    # --------------------------------------------------------

    converted_values = pd.to_numeric(df[column_name],errors = 'coerce')

    # --------------------------------------------------------
    # Identify values that were originally non-null/non-blank
    # but could not be converted to numeric.
    # --------------------------------------------------------

    non_numeric_mask = (converted_values.isna() & ~missing_mask) #Correctly identifies which values were invalid.
                                                                 #Does not include empty or null values
    
    non_numeric_count = non_numeric_mask.sum()

    out_of_range_mask = pd.Series(False,index = df.index)

    if minimum is not None:
        out_of_range_mask = out_of_range_mask | (converted_values<minimum)

    if maximum is not None:
        out_of_range_mask |= converted_values>maximum


    # Do not classify missing/non-numeric values as
    # out-of-range values.

    out_of_range_mask &= converted_values.notna()

    out_of_range_count = out_of_range_mask.sum()

    # --------------------------------------------------------
    # Overall invalid mask
    # --------------------------------------------------------
    
    invalid_mask = non_numeric_mask | out_of_range_mask
    invalid_count = invalid_mask.sum()

    valid_count = len(df) - missing_count - invalid_count

    status = (
        'FAIL'
        if invalid_count>0
        else 'PASS'
    )

    result = {
        "table_name":table_name,
        "column_name":column_name,
        "status":status,
        "total_rows":len(df),
        "missing_values":missing_count,
        "non_numeric_values":non_numeric_count,
        "out_of_range_values":out_of_range_count,
        "invalid_values":invalid_count,
        "valid_values":valid_count,
        "invalid_percentage": round(invalid_count/len(df)*100,2) 
                              if len(df)>0
                              else 0
    }
    return result,invalid_mask


In [24]:
# ============================================================
# RUN NUMERIC VALIDATION
# ============================================================

numeric_validation_results = []
invalid_numeric_records = []

for table_name,column_rules in numeric_rules.items():
    df = datasets[table_name]

    for column_name,rules in column_rules.items():
        result,invalid_mask = validate_numeric_column(
            df = df,
            table_name = table_name,
            column_name = column_name,
            minimum = rules['minimum'],
            maximum = rules['maximum']
        )
        numeric_validation_results.append(result)

        # ----------------------------------------------------
        # Capture complete rows containing invalid values.
        # ----------------------------------------------------

        if invalid_mask.any():
            invalid_rows = df.loc[invalid_mask].copy()

            #Add validation metadata

            invalid_rows['vaildation_table'] = table_name
            invalid_rows['validation_column'] = column_name
            invalid_rows['validation_rule'] = 'invalid_numeric_value'

            invalid_numeric_records.append(invalid_rows)

numeric_validation_report = pd.DataFrame(numeric_validation_results)
numeric_validation_report


,table_name,column_name,status,total_rows,missing_values,non_numeric_values,out_of_range_values,invalid_values,valid_values,invalid_percentage
0,fact_bookings,no_guests,PASS,134590,0,0,0,0,134590,0.00
1,fact_bookings,ratings_given,PASS,134590,77907,0,0,0,56683,0.00
2,fact_bookings,revenue_generated,PASS,134590,0,0,0,0,134590,0.00
3,fact_bookings,revenue_realized,PASS,134590,0,0,0,0,134590,0.00
4,fact_aggregated_bookings,capacity,PASS,9200,0,0,0,0,9200,0.00
5,fact_aggregated_bookings,successful_bookings,PASS,9200,0,0,0,0,9200,0.00


In [25]:
# ============================================================
# QUARANTINE INVALID NUMERIC RECORDS
# ============================================================

def quarantine_invalid_numeric_records(invalid_numeric_records,quarantine_dir):
    """
    Quarantine records containing invalid numeric values.

    Each table/column combination is saved separately.

    The original source DataFrames are never modified.

    Returns
    -------
    all_quarantined_records : pandas.DataFrame
        Combined audit DataFrame containing all quarantined
        numeric records.
    """

    if not invalid_numeric_records:
        print('No invalid records found')
        return pd.DataFrame()

    quarantined_records = []

    for invalid_df in invalid_numeric_records:
        quarantine_df = invalid_df.copy()

        # Retrieve source information

        table_name = quarantine_df['validation_table'].iloc[0]

        column_name = quarantine_df['validation_column'].iloc[0]

        # Add audit information

        quarantine_df['quarantine_reason'] = 'invalid_numeric_value'
        quarantine_df['quarantine_timestamp'] = pd.Timestamp.now()

        # Create a table/column-specific file

        output_file = quarantine_dir/f'{table_name}_{column_name}_invalid_numeric.csv'

        quarantine_df.to_csv(output_file,index = False)

        print(f'{table_name}.{column_name}: {len(quarantine_df):,} records_quarantined')

        print(f'Saved to: {output_file}')

        quarantined_records.append(quarantine_df)

    # --------------------------------------------------------
    # Consolidated audit DataFrame
    # --------------------------------------------------------

    all_quarantined_records = pd.concat(quarantined_records,ignore_index = True)

    print("\n" + "=" * 70)
    print("NUMERIC QUARANTINE COMPLETE")
    print("=" * 70)

    print(f'Total quarantined records: {len(all_quarantined_records):,}')

    return all_quarantined_records

In [26]:
# ============================================================
# RUN NUMERIC QUARANTINE
# ============================================================

quarantined_numeric_records = quarantine_invalid_numeric_records(invalid_numeric_records,QUARANTINE_DIR)

No invalid records found


In [27]:
# ============================================================
# REUSABLE BUSINESS-RULE VALIDATION
# ============================================================

def validate_business_rule(df,table_name,rule_name,invalid_mask):
    """
    Validate a business rule against a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset being validated.

    table_name : str
        Name of the source table.

    rule_name : str
        Business rule being evaluated.

    invalid_mask : pandas.Series
        Boolean Series where True represents a rule violation.

    Returns
    -------
    result : dict
        Summary of the validation result.

    invalid_rows : pandas.DataFrame
        Complete rows that violated the rule.
    """

    # Ensure the mask aligns with the DataFrame
    invalid_mask = invalid_mask.reindex(df.index,fill_value = False)

    invalid_count = invalid_mask.sum()

    result = {
        "table_name": table_name,
        "rule_name": rule_name,
        "total_rows": len(df),
        "invalid_rows": invalid_count,
        "invalid_percentage": round(invalid_count / len(df) * 100,2)
                               if len(df) > 0
                               else 0,
        
        "status":"FAIL"
                  if invalid_count > 0
                  else "PASS"
        }
    # Preserve the entire invalid original row for investigation
    invalid_rows = df.loc[invalid_mask].copy()

    return result,invalid_rows

In [ ]:
# ============================================================
# BUSINESS-RULE VALIDATION
# ============================================================

business_validation_results = []

business_rule_violations = []

# ============================================================
# FACT_BOOKINGS
# ============================================================

booking_df = datasets['fact_bookings']

# ------------------------------------------------------------
# RULE 1
# Booking date cannot be after check-in date.
# Business meaning:
# A customer cannot make a booking after they have already
# checked in.
# ------------------------------------------------------------

invalid_mask = (
    booking_df["booking_date"].notna()
    &
    booking_df["check_in_date"].notna()
    &
    (booking_df["booking_date"]>booking_df["check_in_date"])
)

result,invalid_rows = validate_business_rule(
    booking_df,'fact_bookings','booking_date_after_check_in',invalid_mask
)

business_validation_results.append(result)
if not invalid_rows.empty:
    invalid_rows['validation_table'] = 'fact_bookings'
    invalid_rows['validation_rule'] = 'booking_date_after_check_in'

    business_rule_violations.append(invalid_rows)

# ------------------------------------------------------------
# RULE 2
# Checkout date cannot be before check-in date.
# Business meaning:
# A customer cannot check out before checking in.
# ------------------------------------------------------------

invalid_mask = (
    booking_df['check_in_date'].notna()
    &
    booking_df['checkout_date'].notna()
    &
    (booking_df['checkout_date']<booking_df['check_in_date'])
)

result,invalid_rows = validate_business_rule(
    booking_df,'fact_bookings','checkout_before_check_in',invalid_mask
)    

business_validation_results.append(result)

if not invalid_rows.empty:
    invalid_rows["validation_table"] ="fact_bookings"
    invalid_rows["validation_rule"] = "checkout_before_check_in"
    
    business_rule_violations.append(invalid_rows)

# ------------------------------------------------------------
# RULE 3
# Realized revenue should not exceed generated revenue.
# ------------------------------------------------------------

invalid_mask = (
    booking_df['revenue_realized'].notna()
    &
    booking_df['revenue_generated'].notna()
    &
    (booking_df['revenue_realized']>booking_df['revenue_generated'])
)

result,invalid_rows = validate_business_rule(
    booking_df,'fact_bookings','realized_revenue_exceeds_generated',invalid_mask
)

business_validation_results.append(result)

if not invalid_rows.empty:
    invalid_rows['validation_table'] = 'fact_bookings'
    invalid_rows['validation_rule'] = 'realized_revenue_exceeds_generated'

    business_rule_violations.append(invalid_rows)

# ============================================================
# FACT_AGGREGATED_BOOKINGS
# ============================================================

aggregated_df = datasets['fact_aggregated_bookings']

# ------------------------------------------------------------
# RULE 4
# Successful bookings cannot exceed capacity.
#
# Business meaning:
# Occupied/successful bookings cannot exceed the number of
# rooms available for that property/date/room category.
# ------------------------------------------------------------

invalid_mask = (
    aggregated_df['successful_bookings'].notna()
    &
    aggregated_df['capacity'].notna()
    &
    (aggregated_df['successful_bookings']>aggregated_df['capacity'])
)

result,invalid_rows = validate_business_rule(
    aggregated_df,'fact_aggregated_bookings','successful_bookings_exceed_capacity',invalid_mask
)

business_validation_results.append(result)

if not invalid_rows.empty:
    invalid_rows['validation_table'] = 'fact_aggregated_bookings'
    invalid_rows['validation_rule'] = 'successful_bookings_exceeds_capacity'

    business_rule_violations.append(invalid_rows)

# ============================================================
# CREATE BUSINESS VALIDATION REPORT
# ============================================================

business_validation_report = pd.DataFrame(business_validation_results)

business_validation_report

,table_name,rule_name,total_rows,invalid_rows,invalid_percentage,status
0,fact_bookings,booking_date_after_check_in,134590,0,0.00,PASS
1,fact_bookings,checkout_before_check_in,134590,0,0.00,PASS
2,fact_bookings,realized_revenue_exceeds_generated,134590,0,0.00,PASS
3,fact_aggregated_bookings,successful_bookings_exceed_capacity,9200,0,0.00,PASS


In [ ]:
# ============================================================
# QUARANTINE BUSINESS-RULE VIOLATIONS
# ============================================================

def quarantine_business_rule_violations(violations,quarantine_dir):
    """
    Quarantine complete records that violate business rules.

    Each table + business rule combination is saved as a
    separate CSV file.

    The original datasets are never modified.

    Parameters
    ----------
    violations : list
        List of DataFrames containing business-rule violations.

    quarantine_dir : pathlib.Path
        Directory where quarantine files will be saved.

    Returns
    -------
    all_quarantined_records : pandas.DataFrame
        Consolidated audit DataFrame containing all quarantined
        business-rule violations.
    """

    # --------------------------------------------------------
    # No violations
    # --------------------------------------------------------

    if not violations:
        print('No business rule violations found.')
        return pd.DataFrame()


    quarantined_records = []

    # --------------------------------------------------------
    # Process each table/rule combination
    # --------------------------------------------------------

    for violation_df in violations:
        quarantine_df = violation_df.copy() # Make a copy so we never modify the original

        # ----------------------------------------------------
        # Retrieve metadata added during validation
        # ----------------------------------------------------

        table_name = quarantine_df['validation_table'].iloc[0]
        rule_name = quarantine_df['validation_rule'].iloc[0]

        # ----------------------------------------------------
        # Add audit metadata
        # ----------------------------------------------------

        quarantine_df['quarantine_reason'] = rule_name
        quarantine_df['quarantine_timestamp'] = pd.Timestamp.now()

        # ----------------------------------------------------
        # Create a separate file for each table/rule
        # ----------------------------------------------------

        output_file = quarantine_dir/f'{table_name}_{rule_name}_violations.csv'
        quarantine_df.to_csv(output_file,index = False)

        print(f'{table_name} | {rule_name}: {len(quarantine_df):,} records quarantined')

        print(f'Saved to {output_file}')

        # ----------------------------------------------------
        # Keep a copy for consolidated reporting
        # ----------------------------------------------------

        quarantined_records.append(quarantine_df)

        all_quarantined_records = pd.concat(quarantined_records,ignore_index = True)

        print("\n" + "=" * 70)
        print("BUSINESS-RULE QUARANTINE COMPLETE")
        print("=" * 70)

        print(f'Total qurantined records: {len(all_quarantined_records):,}')

    return all_quarantined_records

In [30]:
# ============================================================
# RUN BUSINESS-RULE QUARANTINE
# ============================================================

quarantined_business_records = quarantine_business_rule_violations(business_rule_violations,QUARANTINE_DIR)

No business rule violations found.


In [31]:
# ============================================================
# DIMENSION TABLE VALIDATION
# ============================================================

# ============================================================
# DIMENSION TABLE CONFIGURATION
# ============================================================

dimension_rules = {
    'dim_hotels': {
        'key_columns': ['property_id'],

    'required_columns': ['property_id','category','city']
    },
    'dim_rooms': {
        'key_columns': ['room_id'],
        'required_columns': ['room_id','room_class']
    },
    'dim_date': {
        'key_columns': ['date'],
        'required_columns': ['date','day_type']
    }
}

In [32]:
# ============================================================
# REUSABLE DIMENSION VALIDATION FUNCTION
# ============================================================

def validate_dimension_table(df,table_name,key_columns,required_columns):
    """
    Validate the structural quality of a dimension table.

    Checks:
        1. Required columns exist
        2. Key columns are not null
        3. Key combinations are unique

    The original DataFrame is never modified.

    Returns
    -------
    result : dict
        Dimension-level validation summary.

    invalid_key_rows : pandas.DataFrame
        Complete rows containing invalid/null or duplicate keys.
    """

    issues = []

    # --------------------------------------------------------
    # Check required columns
    # --------------------------------------------------------

    missing_columns = []
    for column in required_columns:
        if column not in df.columns:
            missing_columns.append(column)

    if missing_columns:
        issues.append(f'missing_columns: {missing_columns}')

    # --------------------------------------------------------
    # If key columns are missing, we cannot perform the
    # remaining key-level checks.
    # --------------------------------------------------------
    
    missing_key_columns = []
    for column in key_columns:
        if column not in df.columns:
            missing_key_columns.append(column)

    if missing_key_columns:
        result = {
            'table_name': table_name,
            'total_rows': len(df),
            'missing_required_columns': len(missing_columns),
            'null_key_rows': None,
            'duplicate_key_rows': None,
            'status': 'FAIL'
        }
        return result,pd.DataFrame

    # --------------------------------------------------------
    # Check null key values
    # --------------------------------------------------------

    null_key_mask = df[key_columns].isna().any(axis=1)

    null_key_count = null_key_mask.sum()

    # --------------------------------------------------------
    # Check duplicate key combinations
    # --------------------------------------------------------

    duplicate_key_mask = df.duplicated(subset = key_columns,keep = False)

    duplicate_key_count = duplicate_key_mask.sum()

    # --------------------------------------------------------
    # Combine key-level failures
    # --------------------------------------------------------

    invalid_key_mask = null_key_mask | duplicate_key_mask

    invalid_key_rows = df.loc[invalid_key_mask].copy()

    # --------------------------------------------------------
    # Determine final status
    # --------------------------------------------------------

    status = (
        'FAIL'
        if (missing_columns or null_key_count>0 or duplicate_key_count>0)
        else 'PASS'
    )

    result = {
                'table_name': table_name,
                'total_rows': len(df),
                'missing_required_columns': len(missing_columns),
                'null_key_rows': null_key_count,
                'duplicate_key_rows': duplicate_key_count,
                'status': status
            }

    return result,invalid_key_rows

In [33]:
# ============================================================
# RUN DIMENSION VALIDATION
# ============================================================

dimension_validation_results = []
dimension_key_violations = []

for table_name,rules in dimension_rules.items():
    result,invalid_rows = validate_dimension_table(
        datasets[table_name],table_name,rules['key_columns'],rules['required_columns']
    )

    dimension_validation_results.append(result)

    # --------------------------------------------------------
    # Capture full records if key-level violations exist.
    # --------------------------------------------------------

    if not invalid_rows.empty:
        invalid_rows['validation_table'] = table_name

        invalid_rows['validation_rule'] = 'dimension_key_validation'

        dimension_key_violations.append(invalid_rows)

dimension_validattion_report = pd.DataFrame(dimension_validation_results)
dimension_validattion_report


,table_name,total_rows,missing_required_columns,null_key_rows,duplicate_key_rows,status
0,dim_hotels,25,0,0,0,PASS
1,dim_rooms,4,0,0,0,PASS
2,dim_date,92,0,0,0,PASS


In [34]:
# ============================================================
# QUARANTINE DIMENSION KEY VIOLATIONS
# ============================================================

def quarantine_dimension_violations(violations,quarantine_dir):
    """
    Quarantine dimension records with invalid keys.

    Each dimension table is saved separately.

    The original data is never modified.
    """

    if not violations:
        print('No dimensions key violations found')
        return pd.DataFrame

    quarantined_records = []

    for violation_df in violations:
        quarantine_df = violation_df.copy()

        table_name = quarantine_df['validation_table'].iloc[0]

        # ----------------------------------------------------
        # Add audit information
        # ----------------------------------------------------

        quarantine_df['quarantine_reason'] = 'dimension_key_validation'
        quarantine_df['quarantine_timestamp'] = pd.Timestamp.mow()

        # ----------------------------------------------------
        # Save source-specific quarantine file
        # ----------------------------------------------------

        output_file = quarantine_dir/f'{table_name}_dimension_key_violations.csv'

        quarantine_df.to_csv(output_file,index = False)

        print(f'{table_name}: {len(quarantine_df):,}')
        print(f'Saved to {output_file}')

        quarantined_records.append(quarantine_df)

        # --------------------------------------------------------
        # Consolidated audit DataFrame
        # --------------------------------------------------------

        all_dimension_violations = pd.concat(quarantined_records,ignore_index = True)

        print("\n" + "=" * 70)
        print("DIMENSION QUARANTINE COMPLETE")
        print("=" * 70)

        print(f'Total quarantined records: {len(all_dimension_violations):,}')

        return all_dimension_violations

In [35]:
# ============================================================
# RUN DIMENSION QUARANTINE
# ============================================================

quarantined_dimension_records = quarantine_dimension_violations(dimension_key_violations,QUARANTINE_DIR)

No dimensions key violations found


#### Data Cleaning

In [36]:
# ============================================================
# BUILD QUARANTINED ROW INDEX REGISTRY
# ===========================================================

def build_quarantine_index_registry(date_records,numeric_records,business_records,dimension_records):
    """
    Build a centralized registry of row indices that should
    be excluded from the cleaned datasets.

    The raw datasets themselves are never modified.

    Returns
    -------
    quarantine_indices : dict
        Dictionary mapping each table name to a set of
        original row indices that must be excluded.
    """

    quarantine_indices = {}

    def register_records(record_list):
        for df in record_list:
            if df.empty:
                continue
            # The validation_table column identifies the
            # original source table.

            if 'validation_table' in df.columns:
                table_name = df['validation_table'].iloc[0]

                quarantine_indices.setdefault(table_name,set())

                quarantine_indices[table_name].update(df.index.tolist())

    # Register all validation failures that should be
    # excluded from the clean output.
    register_records(date_records)
    register_records(numeric_records)
    register_records(business_records)
    register_records(dimension_records)

    return quarantine_indices            

In [37]:
# ============================================================
# CREATE QUARANTINE INDEX REGISTRY
# ============================================================

quarantine_indices = build_quarantine_index_registry(
    invalid_date_records,invalid_numeric_records,business_rule_violations,dimension_key_violations
)
quarantine_indices


{}

In [38]:
# ============================================================
# SAFE TEXT STANDARDIZATION
# ============================================================

def standardize_text_columns(df):
    """
    Standardize text columns without changing their business
    meaning.

    Operations:
        - Remove leading/trailing whitespace
        - Collapse repeated internal whitespace
        - Convert empty/whitespace-only strings to NA

    The original DataFrame is never modified.
    """
    df = df.copy()

    text_columns = df.select_dtypes(include = ['object','string']).columns

    for column in text_columns:
        df[column] = df[column].astype('string').str.strip().str.replace(r'\s+'," ",regex = True) 
        # Collapse repeated internal whitespace

        # Treat empty strings as missing values.
        df[column] = df[column].replace("",pd.NA)

    return df

    

In [39]:
# ============================================================
# DATA TYPE STANDARDIZATION
# ============================================================

def standardize_data_types(df,table_name):
    """
    Convert validated columns to their appropriate data types.

    Invalid records should already have been quarantined before
    this function is applied.
    """

    df = df.copy()

    # --------------------------------------------------------
    # DATE COLUMNS
    # --------------------------------------------------------

    for column in date_columns.get(table_name,[]):
        if column in df.columns:
            df[column] = pd.to_datetime(df[column],errors = 'coerce')

    # --------------------------------------------------------
    # NUMERIC COLUMNS
    # --------------------------------------------------------

    for column in numeric_rules.get(table_name,{}).keys():
        if column in df.columns:
            df[column] = pd.to_numeric(df[column],errors = 'coerce')

    return df

In [40]:
# ============================================================
# BUILD CLEANED DATASETS
# ============================================================

def create_cleaned_datasets(datasets,quarantine_indices):
    """
    Create cleaned versions of all source datasets.

    The raw datasets are never modified.

    Cleaning includes:
        1. Excluding quarantined records
        2. Removing exact duplicate rows
        3. Standardizing text values
        4. Standardizing dates
        5. Standardizing numeric values
    """

    cleaned_datasets = {}

    for table_name,raw_df in datasets.items():
        # ----------------------------------------------------
        # Start from a copy of the raw dataset.
        # ----------------------------------------------------

        df = raw_df.copy()


        # ----------------------------------------------------
        # Remove rows previously classified as invalid.
        # ----------------------------------------------------

        rows_to_remove = quarantine_indices.get(table_name,set())

        if rows_to_remove:
            df = df.loc[~df.index.isin(rows_to_remove)].copy()

        # ----------------------------------------------------
        # Remove completely identical rows.
        #
        # This is safe because every column is identical.
        # We are NOT arbitrarily resolving business-key
        # duplicates here.
        # ----------------------------------------------------

        df = df.drop_duplicates()

        # ----------------------------------------------------
        # Standardize text
        # ----------------------------------------------------

        df = standardize_text_columns(df)

        # ----------------------------------------------------
        # Standardize data types
        # ----------------------------------------------------
        
        df = standardize_data_types(df,table_name)

        # ----------------------------------------------------
        # Store cleaned table
        # ----------------------------------------------------

        cleaned_datasets[table_name] = df

    return cleaned_datasets

In [41]:
# ============================================================
# CREATE CLEANED DATASETS
# ============================================================

cleaned_datasets = create_cleaned_datasets(datasets,quarantine_indices)

print(f'Cleaned {len(cleaned_datasets)} cleaned datasets')

print('\nCleaned tables:')
for table_name in cleaned_datasets:
    print(f' -{table_name}')


Cleaned 5 cleaned datasets

Cleaned tables:
 -fact_bookings
 -fact_aggregated_bookings
 -dim_hotels
 -dim_rooms
 -dim_date


C:\Users\Home\AppData\Local\Temp\ipykernel_12260\2732128346.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column],errors = 'coerce')
C:\Users\Home\AppData\Local\Temp\ipykernel_12260\2732128346.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column],errors = 'coerce')


In [42]:
# ============================================================
# BEFORE / AFTER CLEANING SUMMARY
# ============================================================

cleaning_summary = []

for table_name in datasets:
    raw_count = len(datasets[table_name])

    cleaned_count = len(cleaned_datasets[table_name])

    rows_removed = raw_count - cleaned_count

    removal_percentage = (
        rows_removed/raw_count*100
        if raw_count>0
        else 0
    )

    cleaning_summary.append({
        'table_name': table_name,
        'raw_rows': raw_count,
        'cleaned_rows': cleaned_count,
        'rows_removed': rows_removed,
        'removal_percentage': round(removal_percentage,2)
    })

cleaning_summary = pd.DataFrame(cleaning_summary)

cleaning_summary

,table_name,raw_rows,cleaned_rows,rows_removed,removal_percentage
0,fact_bookings,134590,134590,0,0.00
1,fact_aggregated_bookings,9200,9200,0,0.00
2,dim_hotels,25,25,0,0.00
3,dim_rooms,4,4,0,0.00
4,dim_date,92,92,0,0.00


In [43]:
# ============================================================
# POST-CLEANING DATA TYPES
# ============================================================

for table_name,df in cleaned_datasets.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    display(pd.DataFrame({
        'column_name': df.columns,
        'data_type': df.dtypes.astype(str).values
    }))


TABLE: fact_bookings


,column_name,data_type
0,_source_row_id,int64
1,booking_id,string
2,property_id,int64
3,booking_date,datetime64[ns]
4,check_in_date,datetime64[ns]
5,checkout_date,datetime64[ns]
6,no_guests,int64
7,room_category,string
8,booking_platform,string
9,ratings_given,float64



TABLE: fact_aggregated_bookings


,column_name,data_type
0,_source_row_id,int64
1,property_id,int64
2,check_in_date,datetime64[ns]
3,room_category,string
4,successful_bookings,int64
5,capacity,int64



TABLE: dim_hotels


,column_name,data_type
0,_source_row_id,int64
1,property_id,int64
2,property_name,string
3,category,string
4,city,string



TABLE: dim_rooms


,column_name,data_type
0,_source_row_id,int64
1,room_id,string
2,room_class,string



TABLE: dim_date


,column_name,data_type
0,_source_row_id,int64
1,date,datetime64[ns]
2,mmm yy,string
3,week no,string
4,day_type,string


In [ ]:
# ============================================================
# SAVE CLEANED DATASETS
# ============================================================

def save_cleaned_datasets(cleaned_datasets,output_dir,parquet_dir):
    """
    Save every cleaned dataset to the processed directory.
    """

    saved_files = []

    for table_name,df in cleaned_datasets.items():
        output_file = output_dir/f'{table_name}_clean.csv'
        parquet_file = parquet_dir/f'{table_name}_clean.parquet'

        df.to_csv(output_file,index = False)
        df.to_parquet(parquet_file,index = False)

        saved_files.extend([output_file,parquet_file])

        print(f'Saved CSV: {output_file}')
        print(f'Saved Parquet: {parquet_file}')

    return saved_files

In [ ]:
# ============================================================
# EXPORT CLEANED DATA
# ============================================================

cleaned_files = save_cleaned_datasets(cleaned_datasets,PROCESSED_DATA_DIR,PARQUET_DATA_DIR)

print(f'\nSuccessfully exported {len(cleaned_files)} cleaned datasets.')

Saved: d:\datacleaning_pipeline\data\processed\fact_bookings_clean.csv
Saved: d:\datacleaning_pipeline\data\processed\fact_aggregated_bookings_clean.csv
Saved: d:\datacleaning_pipeline\data\processed\dim_hotels_clean.csv
Saved: d:\datacleaning_pipeline\data\processed\dim_rooms_clean.csv
Saved: d:\datacleaning_pipeline\data\processed\dim_date_clean.csv

Successfully exported 5 cleaned datasets.


In [46]:
def validate_cleaned_dataset(df,table_name):
    """
    Validate a cleaned dataset after the cleaning process.

    This function checks whether known data-quality issues
    remain after cleaning.

    The DataFrame is never modified.

    Returns
    -------
    validation_results : list of dict
        Results for every validation rule.

    """
    # ========================================================
    # 1. EXACT DUPLICATE CHECK
    # ========================================================

    results = []
    duplicate_count = df.duplicated().sum()

    results.append({
        'table_name': table_name,
        'validation_stage': 'post_cleaning',
        'rule_name': 'exact_duplicate_rows',
        'issue_count': duplicate_count,
        'status': 'FAIL'
                   if duplicate_count>0
                   else 'PASS'

    })

    # ========================================================
    # 2. BUSINESS KEY CHECK
    # ========================================================

    if table_name in dimension_rules:
        key_columns = dimension_rules[table_name]['key_columns']

        required_columns = dimension_rules[table_name]['required_columns']

        # ----------------------------------------------------
        # Check required columns
        # ----------------------------------------------------

        missing_required_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        results.append({
            'table_name': table_name,
            'validation_stage': 'post_cleaning',
            'rule_name': 'missing_required_columns',
            'issue_count': len(missing_required_columns),
            'status': 'FAIL'
                       if missing_required_columns
                       else 'PASS'
            })

        # ----------------------------------------------------
        # Check dimension key columns
        # ----------------------------------------------------

        for column in key_columns:
            if column not in df.columns:
                continue

            # Count null/blank key values.

            null_key_mask = df[column].isna() | df[column].astype('string').str.strip().eq("")

            null_key_count = null_key_mask.sum()

            results.append({
                'table_name': table_name,
                'validation_stage': 'post_cleaning',
                'rule_name': f'null_key_{column}',
                'issue_count': null_key_count,
                'status': 'FAIL'
                           if missing_required_columns
                           else 'PASS'
            })

        # ----------------------------------------------------
        # Check duplicate dimension keys
        # ----------------------------------------------------

        existing_key_columns = [
            column
            for column in key_columns
            if column in df.columns
        ]

        if existing_key_columns:
            duplicate_key_mask = df.duplicated(subset = existing_key_columns,keep = False)

            duplicate_key_count = duplicate_key_mask.sum()

            results.append({
                            'table_name': table_name,
                            'validation_stage': 'post_cleaning',
                            'rule_name': 'duplicate_dimension_key',
                            'issue_count': duplicate_key_count,
                            'status': 'FAIL'
                                       if duplicate_key_count>0
                                       else 'PASS'
                        })

    # ========================================================
    # 3. DATE VALIDATION
    # ========================================================

    for column in date_columns.get(table_name,[]):
        # ----------------------------------------------------
        # Check whether expected date column exists.
        # ----------------------------------------------------

        if column not in df.columns:
            results.append({
                "table_name": table_name,
                "validation_stage": "post_cleaning",
                "validation_category": "date",
                "rule_name": f"missing_date_column_{column}",
                "issue_count": 1,
                "status": "FAIL"
            })
            continue

        # ----------------------------------------------------
        # Convert temporarily for validation.
        # ----------------------------------------------------

        converted_dates = pd.to_datetime(df[column],errors = 'coerce')

        # ----------------------------------------------------
        # Treat NaN, empty strings and whitespace as missing.
        # Missing values are NOT automatically invalid.
        # ----------------------------------------------------

        missing_mask = df[column].isna() | df[column].astype('string').str.strip().eq('')

        # ----------------------------------------------------
        # Invalid dates are non-missing values that failed conversion.
        # ----------------------------------------------------

        invalid_mask = (converted_dates.isna() & ~missing_mask)

        invalid_count = invalid_mask.sum()

        results.append({
            "table_name": table_name,
            "validation_stage": "post_cleaning",
            "validation_category": "date",
            "rule_name": f"invalid_date_{column}",
            "issue_count": invalid_count,
            "status": (
                "FAIL"
                if invalid_count > 0
                else "PASS"
            )
        })


    # ========================================================
    # 4. NUMERIC VALIDATION
    # ========================================================

    for column,rules in numeric_rules.get(table_name,{}).items():
        # ----------------------------------------------------
        # Check expected column exists.
        # ----------------------------------------------------

        if column not in df.columns:
            results.append({
                "table_name": table_name,
                "validation_stage": "post_cleaning",
                "validation_category": "numeric",
                "rule_name": f"missing_numeric_column_{column}",
                "issue_count": 1,
                "status": "FAIL"
            })

            continue

        # ----------------------------------------------------
        # Temporary numeric conversion.
        # ----------------------------------------------------

        converted_values = pd.to_numeric(df[column],errors = 'coerce')

        # ----------------------------------------------------
        # Identify missing values.
        # ----------------------------------------------------

        missing_mask = df[column].isna() | df[column].astype('string').str.strip().eq('')

        # ----------------------------------------------------
        # Non-numeric values.
        # ----------------------------------------------------

        non_numeric_mask = converted_values.isna() & ~missing_mask

        # ----------------------------------------------------
        # Range violations.
        # ----------------------------------------------------

        out_of_range_mask = pd.Series(False,index = df.index)

        if rules['minimum'] is not None:
            out_of_range_mask |= converted_values<rules['minimum']

        if rules['maximum'] is not None:
            out_of_range_mask |= converted_values>rules['maximum']

        # Do not treat NaN as an out-of-range value.

        out_of_range_mask &= converted_values.notna()

        invalid_mask = non_numeric_mask | out_of_range_mask

        invalid_count = invalid_mask.sum()

        results.append({
            "table_name": table_name,
            "validation_stage": "post_cleaning",
            "validation_category": "numeric",
            "rule_name": f"invalid_numeric_{column}",
            "issue_count": invalid_count,
            "status": (
                "FAIL"
                if invalid_count > 0
                else "PASS"
            )
        })

    return results

In [47]:
# ============================================================
# RUN POST-CLEANING VALIDATION
# ============================================================

post_cleaning_results = []

for table_name,df in cleaned_datasets.items():
    table_results = validate_cleaned_dataset(df,table_name)

    post_cleaning_results.extend(table_results)

post_cleaning_report = pd.DataFrame(post_cleaning_results)

post_cleaning_report

,table_name,validation_stage,rule_name,issue_count,status,validation_category
0,fact_bookings,post_cleaning,exact_duplicate_rows,0,PASS,NaN
1,fact_bookings,post_cleaning,invalid_date_booking_date,0,PASS,date
2,fact_bookings,post_cleaning,invalid_date_check_in_date,0,PASS,date
3,fact_bookings,post_cleaning,invalid_date_checkout_date,0,PASS,date
4,fact_bookings,post_cleaning,invalid_numeric_no_guests,0,PASS,numeric
5,fact_bookings,post_cleaning,invalid_numeric_ratings_given,0,PASS,numeric
6,fact_bookings,post_cleaning,invalid_numeric_revenue_generated,0,PASS,numeric
7,fact_bookings,post_cleaning,invalid_numeric_revenue_realized,0,PASS,numeric
8,fact_aggregated_bookings,post_cleaning,exact_duplicate_rows,0,PASS,NaN
9,fact_aggregated_bookings,post_cleaning,invalid_date_check_in_date,0,PASS,date


In [48]:
# ============================================================
# POST-CLEANING VALIDATION SUMMARY
# ============================================================

#Puts failed checks towards the top

post_cleaning_report[
    [
        "table_name",
        "validation_category",
        "rule_name",
        "issue_count",
        "status"
    ]
].sort_values(
    by=[
        "status",
        "table_name"
    ],
    ascending=[False, True]
)

,table_name,validation_category,rule_name,issue_count,status
20,dim_date,NaN,exact_duplicate_rows,0,PASS
21,dim_date,NaN,missing_required_columns,0,PASS
22,dim_date,NaN,null_key_date,0,PASS
23,dim_date,NaN,duplicate_dimension_key,0,PASS
24,dim_date,date,invalid_date_date,0,PASS
12,dim_hotels,NaN,exact_duplicate_rows,0,PASS
13,dim_hotels,NaN,missing_required_columns,0,PASS
14,dim_hotels,NaN,null_key_property_id,0,PASS
15,dim_hotels,NaN,duplicate_dimension_key,0,PASS
16,dim_rooms,NaN,exact_duplicate_rows,0,PASS


In [49]:
# ============================================================
# POST-CLEANING FAILURES
# ============================================================

post_cleaning_failures = post_cleaning_report[post_cleaning_report["status"] == "FAIL"].copy()


if post_cleaning_failures.empty:
    print("No post-cleaning validation failures found.")

else:
    print(f"Post-cleaning failures: {len(post_cleaning_failures):,}")

    display(
        post_cleaning_failures[
            [
                "table_name",
                "validation_category",
                "rule_name",
                "issue_count"
            ]
        ]
    )

No post-cleaning validation failures found.


In [50]:
# ============================================================
# DATA QUALITY GATE
# ============================================================

def evaluate_quality_gate(validation_report):
    """
    Determine whether the cleaned datasets passed all
    post-cleaning validation rules.

    Returns
    -------
    bool
        True  -> quality gate passed
        False -> quality gate failed
    """

    failed_checks = validation_report[validation_report['status']=='FAIL']

    if failed_checks.empty:
        print("=" * 70)
        print("DATA QUALITY GATE: PASSED")
        print("=" * 70)
        print("All post-cleaning validation checks passed.")

        return True

    print("=" * 70)
    print("DATA QUALITY GATE: FAILED")
    print("=" * 70)

    print(f"Failed validation checks: {len(failed_checks):,}")

    display(
        failed_checks[
            [
                "table_name",
                "validation_category",
                "rule_name",
                "issue_count"
            ]
        ]
    )

    return False

In [51]:
# ============================================================
# RUN QUALITY GATE
# ============================================================

quality_gate_passed = evaluate_quality_gate(post_cleaning_report)



DATA QUALITY GATE: PASSED
All post-cleaning validation checks passed.


In [52]:
# ============================================================
# BEFORE / AFTER CLEANING IMPACT
# ============================================================

cleaning_impact = []

for table_name in datasets:
    raw_df = datasets[table_name]
    clean_df = datasets[table_name]

    raw_rows = len(raw_df)
    clean_rows = len(clean_df)

    rows_removed = raw_rows - clean_rows
    retention_percentage = (
        clean_rows/raw_rows*100
        if raw_rows>0
        else 0
    )

    cleaning_impact.append({
        "table_name":table_name,
        "raw_rows":raw_rows,
        "cleaned_rows":clean_rows,
        "rows_removed":rows_removed,
        "retention_percentage":round(retention_percentage,2)
    })

cleaning_impact_report = pd.DataFrame(cleaning_impact)

cleaning_impact_report



,table_name,raw_rows,cleaned_rows,rows_removed,retention_percentage
0,fact_bookings,134590,134590,0,100.00
1,fact_aggregated_bookings,9200,9200,0,100.00
2,dim_hotels,25,25,0,100.00
3,dim_rooms,4,4,0,100.00
4,dim_date,92,92,0,100.00


In [53]:
# ============================================================
# PIPELINE QUALITY METRICS
# ============================================================

tables_processed = len(cleaned_datasets)

total_validation_checks = len(post_cleaning_report)

passed_checks = post_cleaning_report['status'].eq('PASS').sum()
failed_checks = post_cleaning_report['status'].eq('FAIL').sum()

total_issues_remaining = post_cleaning_report['issue_count'].fillna(0).sum()

validation_pass_rate = (
    passed_checks/total_validation_checks*100
    if total_validation_checks > 0
    else 0
)

print("=" * 70)
print("PIPELINE QUALITY METRICS")
print("=" * 70)

print(f"Tables processed          : {tables_processed:,}")
print(f"Validation checks         : {total_validation_checks:,}")
print(f"Checks passed             : {passed_checks:,}")
print(f"Checks failed             : {failed_checks:,}")
print(f"Issues remaining          : {total_issues_remaining:,}")
print(f"Validation pass rate      : {validation_pass_rate:.2f}%")
print(f"Quality gate              : {'PASSED' if quality_gate_passed else 'FAILED'}")

PIPELINE QUALITY METRICS
Tables processed          : 5
Validation checks         : 25
Checks passed             : 25
Checks failed             : 0
Issues remaining          : 0
Validation pass rate      : 100.00%
Quality gate              : PASSED


In [54]:
# ============================================================
# EXPORT POST-CLEANING REPORTS
# ============================================================

# Save detailed validation results.

post_cleaning_report.to_csv(REPORTS_DIR/'post_cleaning_validation.csv',index = False)

# Save before/after cleaning impact.

cleaning_impact_report.to_csv(REPORTS_DIR/'cleaning_impact.csv',index = False)

print('Post-cleaning reports exported successfully.')
print(f"Validation report: {REPORTS_DIR / 'post_cleaning_validation.csv'}")
print(f"Cleaning impact: {REPORTS_DIR / 'cleaning_impact.csv'}")

Post-cleaning reports exported successfully.
Validation report: d:\datacleaning_pipeline\reports\post_cleaning_validation.csv
Cleaning impact: d:\datacleaning_pipeline\reports\cleaning_impact.csv


In [55]:
# ============================================================
# FINAL DATA QUALITY SCORECARD
# ============================================================

# ------------------------------------------------------------
# Calculate final metrics
# ------------------------------------------------------------

tables_processed = len(cleaned_datasets)

raw_rows_processed = sum(
    len(df)
    for df in datasets.values()
)

clean_rows_output = sum(
    len(df) 
    for df in cleaned_datasets.values()
)

rows_removed = raw_rows_processed - clean_rows_output

records_quarantined = sum(
    len(row_indices)
    for row_indices in quarantine_indices.values()
)

validation_checks = len(post_cleaning_report)
validation_checks_passed = post_cleaning_report['status'].eq('PASS').sum()

validation_checks_failed = post_cleaning_report['status'].eq('FAIL').sum()

validation_pass_rate = (
    validation_checks_passed/validation_checks*100
    if validation_checks>0
    else 0
)

data_retention_rate = (
    clean_rows_output/raw_rows_processed*100
    if raw_rows_processed>0
    else 0
)

quality_gate = (
    'PASSED'
    if validation_checks_failed == 0
    else 'FAILED'
)

# ------------------------------------------------------------
# Create clean presentation scorecard
# ------------------------------------------------------------

quality_scorecard = pd.DataFrame({
    "Metric": [
        "Tables Processed",
        "Raw Records Processed",
        "Clean Records Output",
        "Records Removed",
        "Records Quarantined",
        "Validation Checks",
        "Checks Passed",
        "Checks Failed",
        "Validation Pass Rate",
        "Data Retention Rate",
        "Quality Gate"
    ],

    "Result": [
        f"{tables_processed:,}",
        f"{raw_rows_processed:,}",
        f"{clean_rows_output:,}",
        f"{rows_removed:,}",
        f"{records_quarantined:,}",
        f"{validation_checks:,}",
        f"{validation_checks_passed:,}",
        f"{validation_checks_failed:,}",
        f"{validation_pass_rate:.2f}%",
        f"{data_retention_rate:.2f}%",
        quality_gate
    ]
})


quality_scorecard

,Metric,Result
0,Tables Processed,5
1,Raw Records Processed,"143,911"
2,Clean Records Output,"143,911"
3,Records Removed,0
4,Records Quarantined,0
5,Validation Checks,25
6,Checks Passed,25
7,Checks Failed,0
8,Validation Pass Rate,100.00%
9,Data Retention Rate,100.00%


In [56]:
# ============================================================
# EXPORT QUALITY SCORECARD
# ============================================================

quality_scorecard.to_csv(REPORTS_DIR/'quality_scorecard.csv',index = False)

print('Quality exported successfully.')
print(f"Quality report: {REPORTS_DIR / 'quality_scorecard.csv'}")

Quality exported successfully.
Quality report: d:\datacleaning_pipeline\reports\quality_scorecard.csv
